# Importing LAS Files

Log ASCII Standard (LAS) is a widely used file format in the oil and gas industry for storing well logging data, developed by the Canadian Well Logging Society (CWLS). The links below provide PDF versions of the original documents where you can learn about the LAS standards:

- [LAS 2.0](https://www.cwls.org/wp-content/uploads/2017/02/Las2_Update_Feb2017.pdf)
- [LAS 3.0](https://www.cwls.org/wp-content/uploads/2014/09/LAS_3_File_Structure.pdf)

RockVerse has a fully-implemented module to import Well data from LAS files.

## The Las class

To import a LAS file content into RockVerse, you just need to call the top-level `read_las` function:

```python
import rockverse as rv
las_data = rv.read_las('/path/to/your/las/file.las')
```

For demonstrative purposes, the LAS samples in the original documents referenced above are hard-coded in RockVerse. Let's import and explore the LAS 3.0 sample data:

In [1]:
import rockverse as rv
las_data = rv.cwls_las_sample(version=3, sample=1)

The new object will be an instance of the [Las class](../../../api/las/lasclass.rst):

In [2]:
las_data

The LAS version in the imported LAS file is stored in the `version` attribute. If the first lines in the file are comment lines, they will be stored in the `initial_comments` attribute.

In [3]:
print(f"LAS version: {las_data.version}\n")
print(f"Initial comments:\n-----------------\n{las_data.initial_comments}")

LAS version: 3

Initial comments:
-----------------
 from https://www.cwls.org/wp-content/uploads/2014/09/LAS_3_File_Structure.pdf
 Note! Ver 3.0 specifications are preliminary! Do not assume they will not change.
 This is strictly an example to illustrate the proposed changes!



Let's see what is inside by calling the `tree` method:

In [4]:
las_data.tree()

 from https://www.cwls.org/wp-content/uploads/2014/09/LAS_3_File_Structure.pdf
 Note! Ver 3.0 specifications are preliminary! Do not assume they will not change.
 This is strictly an example to illustrate the proposed changes!



|- Well
|   |-[0] STRT: 1660.125 M (First Index Value)
|   |-[1] STOP: 1660.875 M (Last Index Value)
|   |-[2] STEP: 0.125 M (STEP)
|   |-[3] NULL: -999.25 (NULL VALUE)
|   |-[4] COMP: ANY OIL COMPANY INC. (COMPANY)
|   |-[5] WELL: ANY ET AL 01-02-03-04 (WELL)
|   |-[6] FLD: WILDCAT (FIELD)
|   |-[7] LOC: 1-2-3-4W5M (LOCATION)
|   |-[8] SRVC: ANY LOGGING COMPANY INC. (SERVICE COMPANY)
|   |-[9] DATE: 1986-12-13 (Service DATE)
|   |-[10] CTRY: CA (COUNTRY)
|   |-[11] PROV: ALBERTA (PROVINCE)
|   |-[12] UWI: 100010200304W500 (UNIQUE WELL ID)
|   |-[13] LIC: 123456 (LICENSE NUMBER)
|   |-[14] LATI: 45.37° 12' 58" (X LOCATION)
|   |-[15] LONG: 13.22° 30' 09" (Y LOCATION)
|   |-[16] GDAT: NAD83 (Geodetic Datum)
|- Curve
|   |- parameters:
|   |   |-[0] PDAT: GL (Pe

Note the tree indentation levels. Top level is the section name (`Well`, `Curve`, etc.), second level is the `parameter` or `data` subsection (except for the `Well` section, which is a parameter-like section), and third level are the corresponding data entries. The original entry order from the LAS file is kept and shown by the numbers inside the brackets.

The LAS object supports key-based access using square brackets ([]), which will retrieve the corresponding section as a [LasSection class](../../../api/las/lassectionclass.rst) object
(except for the `Well` section, as we mentioned above...):


In [5]:
las_data['Well']

In [6]:
las_data['Curve']

In [7]:
las_data['Perforation_Data']

The section names are gathered by the `section_keys()` method:

In [8]:
las_data.section_keys()

dict_keys(['Well', 'Curve', 'Drilling_Data', 'Core_Data[1]', 'Core_Data[2]', 'Inclinometry_Data', 'Test_Data', 'Tops_Data', 'Perforation_Data'])

There is also a `find` method that will filter the `tree` output with mnemonics matching a pattern using Unix shell-style wildcards:

In [9]:
las_data.find('D*')



|- Well
|   |-[9] DATE: 1986-12-13 (Service DATE)
|- Curve
|   |- parameters:
|   |   |-[2] DREF: KB (Depth Reference (KB,DF,CB))
|   |   |-[12] DMAT_Depth[1]: (523.0, 1510.0) M (Density Matrix Depth interval)
|   |   |-[13] DMAT_Depth[2]: (1510.0, 2510.0) M (Density Matrix Depth interval)
|   |- data:
|   |   |-[0] DEPT, M: DEPTH
|   |   |-[1] DT, US/M (123 456 789): SONIC TRANSIT TIME
|   |   |-[2] DPHI, V/V (123 456 789): DENSITY POROSITY | ('MDEN[1]', 'MDEN[2]')
|- Drilling_Data
|   |- data:
|   |   |-[0] DEPT, ft: Depth
|   |   |-[1] DIST, ft: Cumulative increment of drilling.
|- Inclinometry_Data
|   |- data:
|   |   |-[3] DEVI, DEG: Borehole Deviation
|- Test_Data
|   |- data:
|   |   |-[3] DDES: TEST Recovery Description


## Accessing sections

As seen above, each value retrieved by key-based access is a [LasSection class](../../../api/las/lassectionclass.rst) object ("well", you know...)

In [10]:
las_data["Well"]

In [11]:
las_data['Curve']

You can use the `tree` and `find` methods in `LasSection`'s as well:

In [12]:
las_data['Curve'].tree()


|- parameters:
|   |-[0] PDAT: GL (Permanent Data)
|   |-[1] APD: 4.2 M (Above Permanent Data)
|   |-[2] DREF: KB (Depth Reference (KB,DF,CB))
|   |-[3] EREF: 234.5 M (Elevation of Depth Reference)
|   |-[4] RUN: 1 (Run Number)
|   |-[5] RUNS: 2 (# of Runs for this well.)
|   |-[6] RUN[1]: 2 (Number of the indexed RUN)
|   |-[7] RUN[2]: 3 (Number of the indexed RUN)
|   |-[8] RUN_Depth[1]: (0.0, 1500.0) M (Run 1 Depth Interval)
|   |-[9] RUN_Depth[2]: (1500.0, 2513.0) M (Run 2 Depth Interval)
|   |-[10] NMAT_Depth[1]: (523.0, 1500.0) M (Neutron Matrix Depth interval)
|   |-[11] NMAT_Depth[2]: (1500.0, 2500.0) M (Neutron Matrix Depth interval)
|   |-[12] DMAT_Depth[1]: (523.0, 1510.0) M (Density Matrix Depth interval)
|   |-[13] DMAT_Depth[2]: (1510.0, 2510.0) M (Density Matrix Depth interval)
|   |-[14] MATR[1]: SAND (Neutron Porosity Matrix)
|   |-[15] MATR[2]: LIME (Neutron Porosity Matrix)
|   |-[16] MDEN[1]: 2650 KG/M3 (Neutron Porosity Matrix)
|   |-[17] MDEN[2]: 2710 KG/M3 (Neut

In [13]:
las_data['Curve'].find('*R[*')


|- parameters:
|   |-[14] MATR[1]: SAND (Neutron Porosity Matrix)
|   |-[15] MATR[2]: LIME (Neutron Porosity Matrix)
|   |-[18] FR_LR[1]: 500,100 M
|   |-[19] FR_LR[2]: 523,100 M
|   |-[20] FR_LR[3]: 520,100 M
|   |-[21] FR_LR[4]: 500,100 M
|   |-[22] FR_LR[5] M
|   |-[23] FR_LR[6]: 510,100 M
|   |-[24] FR_LR[7]: 510,100 M
|   |-[25] FR_LR[8]: 510,100 M
|   |-[26] FR_LR[9]: 510,100 M
|   |-[27] FR_LR[10]: 510,100 M
|- data:
|   |-[6] NMR[1], mv (123 456 789): NMR Echo Array
|   |-[7] NMR[2], mv (123 456 789): NMR Echo Array
|   |-[8] NMR[3], mv (123 456 789): NMR Echo Array
|   |-[9] NMR[4], mv (123 456 789): NMR Echo Array
|   |-[10] NMR[5], mv (123 456 789): NMR Echo Array


## Accessing Parameters and data

Parameters and data are accessed by the corresponding keys or attributes, and will return instances of 
[LasParam](../../../api/las/lasparameterclass.rst) and
[LasData](../../../api/las/lasdataclass.rst),
respectively:


In [14]:
las_data['Curve']['parameters']

In [15]:
las_data['Curve'].parameters

In [16]:
las_data['Curve']['data']

In [17]:
las_data['Curve'].data

These classes also support `tree` and `find` methods:

In [18]:
las_data['Curve'].parameters.tree()


|-[0] PDAT: GL (Permanent Data)
|-[1] APD: 4.2 M (Above Permanent Data)
|-[2] DREF: KB (Depth Reference (KB,DF,CB))
|-[3] EREF: 234.5 M (Elevation of Depth Reference)
|-[4] RUN: 1 (Run Number)
|-[5] RUNS: 2 (# of Runs for this well.)
|-[6] RUN[1]: 2 (Number of the indexed RUN)
|-[7] RUN[2]: 3 (Number of the indexed RUN)
|-[8] RUN_Depth[1]: (0.0, 1500.0) M (Run 1 Depth Interval)
|-[9] RUN_Depth[2]: (1500.0, 2513.0) M (Run 2 Depth Interval)
|-[10] NMAT_Depth[1]: (523.0, 1500.0) M (Neutron Matrix Depth interval)
|-[11] NMAT_Depth[2]: (1500.0, 2500.0) M (Neutron Matrix Depth interval)
|-[12] DMAT_Depth[1]: (523.0, 1510.0) M (Density Matrix Depth interval)
|-[13] DMAT_Depth[2]: (1510.0, 2510.0) M (Density Matrix Depth interval)
|-[14] MATR[1]: SAND (Neutron Porosity Matrix)
|-[15] MATR[2]: LIME (Neutron Porosity Matrix)
|-[16] MDEN[1]: 2650 KG/M3 (Neutron Porosity Matrix)
|-[17] MDEN[2]: 2710 KG/M3 (Neutron Porosity Matrix)
|-[18] FR_LR[1]: 500,100 M
|-[19] FR_LR[2]: 523,100 M
|-[20] FR_LR

In [19]:
las_data['Curve'].parameters.find('DMAT*')


|-[12] DMAT_Depth[1]: (523.0, 1510.0) M (Density Matrix Depth interval)
|-[13] DMAT_Depth[2]: (1510.0, 2510.0) M (Density Matrix Depth interval)


In [20]:
las_data['Curve'].data.tree()


|-[0] DEPT, M: DEPTH
|-[1] DT, US/M (123 456 789): SONIC TRANSIT TIME
|-[2] DPHI, V/V (123 456 789): DENSITY POROSITY | ('MDEN[1]', 'MDEN[2]')
|-[3] NPHI, V/V (123 456 789): NEUTRON POROSITY | ('MATR[1]', 'MATR[2]')
|-[4] YME, PA (123 456 789): YOUNGS MODULES
|-[5] CDES (123 456 789): CORE DESCRIPTION
|-[6] NMR[1], mv (123 456 789): NMR Echo Array
|-[7] NMR[2], mv (123 456 789): NMR Echo Array
|-[8] NMR[3], mv (123 456 789): NMR Echo Array
|-[9] NMR[4], mv (123 456 789): NMR Echo Array
|-[10] NMR[5], mv (123 456 789): NMR Echo Array


In [21]:
las_data['Curve'].data.find('*PHI')


|-[2] DPHI, V/V (123 456 789): DENSITY POROSITY | ('MDEN[1]', 'MDEN[2]')
|-[3] NPHI, V/V (123 456 789): NEUTRON POROSITY | ('MATR[1]', 'MATR[2]')


You also have key-based access using square brackets, either by integer indices or by mnemonic, retrieving the parameter or data values as a dictionary:

In [22]:
las_data['Curve'].parameters[5]

{'mnem': 'RUNS',
 'unit': '',
 'value': 2,
 'description': '# of Runs for this well.',
 'format': 'I',
 'association': []}

In [23]:
las_data['Curve'].parameters['DREF']

{'mnem': 'DREF',
 'unit': '',
 'value': 'KB',
 'description': 'Depth Reference (KB,DF,CB)',
 'format': '',
 'association': []}

In [24]:
las_data['Curve'].data[1]

{'mnem': 'DT',
 'unit': 'US/M',
 'description': 'SONIC TRANSIT TIME',
 'format': 'F',
 'association': [],
 'code': '123 456 789',
 'value': array([123.45, 123.45, 123.45, 123.45, 123.45, 123.45, 123.45])}

In [25]:
las_data['Curve'].data['DPHI']

{'mnem': 'DPHI',
 'unit': 'V/V',
 'description': 'DENSITY POROSITY',
 'format': 'F',
 'association': ('MDEN[1]', 'MDEN[2]'),
 'code': '123 456 789',
 'value': array([0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17])}

## Accessing data using forward slash ('/') notation

RockVerse LAS objects support accessing nested sections and subsections using a forward slash (`/`) notation. This allows you to directly retrieve parameters or data entries by specifying the full path in the LAS hierarchy.

For example, you can access the `parameters` subsection inside the `Curve` section, or even a specific parameter within it, using keys like `"Curve/parameters"` or `"Curve/parameters/DPHI"`.

In [26]:
las_data['Curve/parameters']

In [27]:
las_data['Curve/parameters/EREF']

{'mnem': 'EREF',
 'unit': 'M',
 'value': 234.5,
 'description': 'Elevation of Depth Reference',
 'format': '',
 'association': []}

In [28]:
las_data['Curve']['parameters/EREF']

{'mnem': 'EREF',
 'unit': 'M',
 'value': 234.5,
 'description': 'Elevation of Depth Reference',
 'format': '',
 'association': []}

This notation simplifies navigation of complex LAS files by providing intuitive key-based access.


Similar to the `find` method, the `find_path` method allows you to search for mnemonics matching a Unix shell-style wildcard pattern and returns their full paths within the LAS file hierarchy using forward slash (`/`) notation:

In [29]:
las_data.find_path('*NMR*')

['Curve/data/NMR[1]',
 'Curve/data/NMR[2]',
 'Curve/data/NMR[3]',
 'Curve/data/NMR[4]',
 'Curve/data/NMR[5]']

Every hierarchical level in the LAS object model implements this method, allowing flexible and intuitive searches at any depth. The result is always a list containing the full paths for the matching mnemonics, relative to the calling object. 

In [30]:
obj = las_data['Curve']
obj.find_path('*NMR*')

['data/NMR[1]', 'data/NMR[2]', 'data/NMR[3]', 'data/NMR[4]', 'data/NMR[5]']

In [31]:
obj = las_data['Curve/data']
obj.find_path('*NMR*')

['NMR[1]', 'NMR[2]', 'NMR[3]', 'NMR[4]', 'NMR[5]']

This is particularly useful to quickly locate parameters or data entries across all sections and subsections.

## Importing into basic data types

The main purpose of RockVerse `Las` objects is to provide a structured representation of LAS files and mediate the import of LAS information into native data types, which are optimized for high performance processing, analysis, and visualization.

### Parallel arrays and coordinates

Each entry in a data subsection can be exported as a parallel array or a coordinate (provided the assumptions for a coordinate array are met):

In [32]:
DT = las_data['Curve/data/DT'].as_parallelarray()

# Let's see what is inside...
print(f"""
Imported data:
    type: {type(DT)}\n
    values:
        {DT[...]}\n
    attrs:\n        {'\n       '.join(str(DT.attrs.asdict())[1:-1].split(','))}
""")


Imported data:
    type: <class 'rockverse.core.parallelarray.ParallelArray'>

    values:
        [123.45 123.45 123.45 123.45 123.45 123.45 123.45]

    attrs:
        '_ROCKVERSE_DATATYPE': 'ParallelArray'
        'name': 'DT'
        'unit': 'US/M'
        'description': 'SONIC TRANSIT TIME'
        'code': '123 456 789'



In [33]:
DEPT = las_data['Curve/data/DEPT'].as_coordinate()

# Let's see what is inside...
print(f"""
Imported data:
    type: {type(DEPT)}\n
    values:
        {DEPT[...]}\n
    attrs:\n        {'\n       '.join(str(DEPT.attrs.asdict())[1:-1].split(','))}
""")


Imported data:
    type: <class 'rockverse.core.coordinates.Coordinate'>

    values:
        [1660.125 1660.25  1660.375 1660.5   1660.625 1660.75  1660.875]

    attrs:
        '_ROCKVERSE_DATATYPE': 'Coordinate'
        'name': 'DEPT'
        'unit': 'M'
        'description': 'DEPTH'



### Scalar fields and field groups

Individual scalar fields can be built by simply calling the class constructor with the exported array and coordinates:

In [34]:
DT_log = rv.ScalarField(array=DT, coords=(DEPT,))

print(f"""
Imported data:
    type: {type(DT_log)}

    array values:
        {DT_log.array[...]}

    array attrs:\n        {'\n       '.join(str(DT_log.array.attrs.asdict())[1:-1].split(','))}

    coord values:
        {DT_log.coords[0][...]}

    coord attrs:\n        {'\n       '.join(str(DT_log.coords[0].attrs.asdict())[1:-1].split(','))}

""")


Imported data:
    type: <class 'rockverse.core.scalarfield.ScalarField'>

    array values:
        [123.45 123.45 123.45 123.45 123.45 123.45 123.45]

    array attrs:
        '_ROCKVERSE_DATATYPE': 'ParallelArray'
        'name': 'DT'
        'unit': 'US/M'
        'description': 'SONIC TRANSIT TIME'
        'code': '123 456 789'

    coord values:
        [1660.125 1660.25  1660.375 1660.5   1660.625 1660.75  1660.875]

    coord attrs:
        '_ROCKVERSE_DATATYPE': 'Coordinate'
        'name': 'DEPT'
        'unit': 'M'
        'description': 'DEPTH'




but you can export the whole section at once as field group:

In [35]:
# Import 'Inclinometry_Data' data 'AZIM' into a scalar field
log_group = las_data['Inclinometry_Data/data'].create_fieldgroup(coordinate_column='TVD')
log_group.array_keys


('MD', 'DEVI', 'AZIM', 'RB')

In [36]:

'''
# Let's see what is inside...
print(f"""
Imported data:
    type: {type(field)}

    array values:
        {field.array[...]}

    array attrs:\n        {'\n       '.join(str(field.array.attrs.asdict())[1:-1].split(','))}

    coord values:
        {field.coords[0][...]}

    coord attrs:\n        {'\n       '.join(str(field.coords[0].attrs.asdict())[1:-1].split(','))}

""")
'''

'\n# Let\'s see what is inside...\nprint(f"""\nImported data:\n    type: {type(field)}\n\n    array values:\n        {field.array[...]}\n\n    array attrs:\n        {\'\n       \'.join(str(field.array.attrs.asdict())[1:-1].split(\',\'))}\n\n    coord values:\n        {field.coords[0][...]}\n\n    coord attrs:\n        {\'\n       \'.join(str(field.coords[0].attrs.asdict())[1:-1].split(\',\'))}\n\n""")\n'



`LasData` well log-type entries can be directly imported into RockVerse 
[scalar fields](../../../api/core/scalarfield.rst) using the `create_scalarfield` method.
All you need is to inform the column index or mnemonics, and the importing method will take care of assigning coordinates and attributes:

In [37]:
# Import 'Inclinometry_Data' data 'AZIM' into a scalar field
field = las_data['Inclinometry_Data/data'].create_scalarfield(column='AZIM')

# Let's see what is inside...
print(f"""
Imported data:
    type: {type(field)}

    array values:
        {field.array[...]}

    array attrs:\n        {'\n       '.join(str(field.array.attrs.asdict())[1:-1].split(','))}

    coord values:
        {field.coords[0][...]}

    coord attrs:\n        {'\n       '.join(str(field.coords[0].attrs.asdict())[1:-1].split(','))}

""")

AttributeError: 'LasData' object has no attribute 'create_scalarfield'

Note the first column `MD` is automatically assigned as the coordinate object. You can pass any other column as the coordinate object using the `coordinate_column` parameter:

In [ ]:
field = las_data['Inclinometry_Data/data'].create_scalarfield(column='AZIM', coordinate_column='TVD')

print(f"""
Imported data:
    type: {type(field)}

    array values:
        {field.array[...]}

    array attrs:\n        {'\n       '.join(str(field.array.attrs.asdict())[1:-1].split(','))}

    coord values:
        {field.coords[0][...]}

    coord attrs:\n        {'\n       '.join(str(field.coords[0].attrs.asdict())[1:-1].split(','))}

""")

You can also pass the column integer index instead of the mnemonics:

In [ ]:
field = las_data['Inclinometry_Data/data'].create_scalarfield(column=2)

print(f"""
Imported data:
    type: {type(field)}

    array values:
        {field.array[...]}

    array attrs:\n        {'\n       '.join(str(field.array.attrs.asdict())[1:-1].split(','))}

    coord values:
        {field.coords[0][...]}

    coord attrs:\n        {'\n       '.join(str(field.coords[0].attrs.asdict())[1:-1].split(','))}

""")